In [2]:
import pandas as pd
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import nltk
import re
import numpy as np



# Load data
df = pd.read_csv("/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv")  # replace with your actual path
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)
# Drop rows with missing reviews or ratings
df = df.dropna(subset=['review', 'rating'])
df.drop(['patient_id', 'condition'], axis=1, inplace=True)

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

def clean_text(text):
    # Lowercase, remove punctuation and digits
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text
df['review_clean'] = df['review'].astype(str).apply(clean_text)
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['review_clean'])



In [4]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


X_train, X_test, y_train, y_test = train_test_split(X_bow, df['rating'], test_size=0.2, random_state=42)

lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

MSE: 9.393897268231854


Train & Test from Clean Part

In [5]:


train_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv')
test_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

train_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)
test_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)

vectorizer = CountVectorizer(max_features=60000)  # Limit features if needed
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values


lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train_bow, y_train)

y_pred = lasso.predict(X_test_bow)
print("Test MSE:", mean_squared_error(y_test, y_pred))

Test MSE: 9.806323542452168


In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

# === 1. Load data ===
train_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv')
test_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

# Drop unneeded columns
train_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)
test_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)

# === 2. Feature Extraction (BoW ) ===
vectorizer = CountVectorizer(max_features=60000)
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values
vocab = vectorizer.get_feature_names_out()

In [12]:
# === LASSO for Sparse Interpretability ===
lasso = Lasso(alpha=0.01, max_iter=1000)
lasso.fit(X_train_bow, y_train)
y_pred_lasso = lasso.predict(X_test_bow)

print("Test MSE with LASSO:", mean_squared_error(y_test, y_pred_lasso))

nonzero_idx = np.where(lasso.coef_ != 0)[0]
nonzero_weights = lasso.coef_[nonzero_idx]
nonzero_words = vectorizer.get_feature_names_out()[nonzero_idx]

print(nonzero_weights.shape)

top_positive = sorted(zip(nonzero_words, nonzero_weights), key=lambda x: -x[1])[:15]
top_negative = sorted(zip(nonzero_words, nonzero_weights), key=lambda x: x[1])[:15]


print("\nTop Positive Influential Words:")
for word, weight in top_positive:
    print(f"{word}: {weight:.3f}")

print("\nTop Negative Influential Words:")
for word, weight in top_negative:
    print(f"{word}: {weight:.3f}")


Test MSE with LASSO: 7.7327798768489275
(205,)

Top Positive Influential Words:
miracle: 1.287
love: 1.002
amazing: 0.993
highly: 0.710
wonderful: 0.688
happy: 0.654
great: 0.626
save: 0.552
thank: 0.514
life: 0.498
good: 0.462
year: 0.407
clear: 0.385
easy: 0.382
free: 0.379

Top Negative Influential Words:
horrible: -0.949
depressed: -0.755
remove: -0.719
awful: -0.687
call: -0.616
terrible: -0.578
bad: -0.576
waste: -0.574
bleed: -0.564
switch: -0.527
cry: -0.526
stop: -0.516
constantly: -0.483
ruin: -0.460
cause: -0.455


In [22]:
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import numpy as np


# Step 1: Reduce BoW features to 10 PCA dimensions
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_bow)
X_test_pca = pca.transform(X_test_bow)

# Step 2: Train Lasso on reduced features
lasso = Lasso(alpha=0.01, max_iter=1000)
lasso.fit(X_train_pca, y_train)
y_pred_lasso = lasso.predict(X_test_pca)

# Step 3: Evaluate
print("Test MSE with LASSO + PCA:", mean_squared_error(y_test, y_pred_lasso))

# Step 4: Interpret PCA components
feature_names = vectorizer.get_feature_names_out()
for i, component in enumerate(pca.components_):
    print(f"\nPCA Component {i + 1}:")
    top_features = sorted(
        zip(feature_names, component), key=lambda x: -abs(x[1])
    )[:10]
    for word, weight in top_features:
        print(f"{word}: {weight:.3f}")



Test MSE with LASSO + PCA: 10.118351706022446

PCA Component 1:
day: 0.632
take: 0.302
period: 0.266
feel: 0.244
pill: 0.227
start: 0.224
month: 0.219
week: 0.200
get: 0.163
go: 0.146

PCA Component 2:
month: 0.400
period: 0.394
day: -0.312
mg: -0.291
feel: -0.235
pill: 0.222
take: -0.204
get: 0.167
birth: 0.161
control: 0.161

PCA Component 3:
day: 0.574
feel: -0.408
take: -0.239
pain: 0.235
mg: -0.207
start: -0.203
month: -0.195
year: -0.183
anxiety: -0.174
like: -0.154

PCA Component 4:
take: 0.698
feel: -0.495
pain: 0.229
like: -0.192
pill: 0.184
day: -0.163
year: 0.140
week: -0.131
work: 0.100
mg: 0.073

PCA Component 5:
pain: 0.817
year: 0.200
take: -0.193
month: 0.164
day: -0.152
get: 0.150
feel: 0.149
go: 0.143
bad: 0.130
pill: -0.102

PCA Component 6:
mg: 0.425
year: 0.421
feel: -0.398
take: -0.340
month: 0.228
pill: -0.216
day: 0.208
like: -0.187
start: 0.163
pain: -0.134

PCA Component 7:
week: 0.588
year: -0.563
start: 0.268
pain: 0.188
feel: -0.178
month: 0.155
pill: -0.12

In [21]:
feature_names = vectorizer.get_feature_names_out()

for i, component in enumerate(pca.components_):
    top_indices = np.argsort(-np.abs(component))[:10]
    top_words = [(feature_names[j], component[j]) for j in top_indices]
    print(f"\nPCA Component {i} - Candidate Topic:")
    for word, weight in top_words:
        print(f"{word}: {weight:.3f}")


PCA Component 0 - Candidate Topic:
day: 0.632
take: 0.302
period: 0.266
feel: 0.244
pill: 0.227
start: 0.224
month: 0.219
week: 0.200
get: 0.163
go: 0.146

PCA Component 1 - Candidate Topic:
month: 0.400
period: 0.394
day: -0.312
mg: -0.291
feel: -0.235
pill: 0.222
take: -0.204
get: 0.167
birth: 0.161
control: 0.161

PCA Component 2 - Candidate Topic:
day: 0.574
feel: -0.408
take: -0.239
pain: 0.235
mg: -0.207
start: -0.203
month: -0.195
year: -0.183
anxiety: -0.174
like: -0.154

PCA Component 3 - Candidate Topic:
take: 0.698
feel: -0.495
pain: 0.229
like: -0.192
pill: 0.184
day: -0.163
year: 0.140
week: -0.131
work: 0.100
mg: 0.073

PCA Component 4 - Candidate Topic:
pain: 0.817
year: 0.200
take: -0.193
month: 0.164
day: -0.152
get: 0.150
feel: 0.149
go: 0.143
bad: 0.130
pill: -0.102

PCA Component 5 - Candidate Topic:
mg: 0.425
year: 0.421
feel: -0.398
take: -0.340
month: 0.228
pill: -0.216
day: 0.208
like: -0.187
start: 0.163
pain: -0.134

PCA Component 6 - Candidate Topic:
week: 0

In [57]:
# === Inspect Top Words in PCA Dimensions ===
def print_top_words_per_component(svd, vocab, n_words=10, n_components=5):
    print("\nTop words per PCA component:")
    for i in range(n_components):
        comp = svd.components_[i]
        top_indices = np.argsort(np.abs(comp))[-n_words:][::-1]
        words = [vocab[j] for j in top_indices]
        print(f"Component {i + 1}: {', '.join(words)}")

print_top_words_per_component(svd, vocab)



Top words per PCA component:
Component 1: day, take, feel, month, year, start, week, work, go, time
Component 2: period, month, mg, pill, day, feel, get, birth, control, take
Component 3: day, year, period, month, work, effect, mg, anxiety, pill, try
Component 4: pain, feel, year, like, week, take, work, start, anxiety, doctor
Component 5: take, pain, feel, pill, like, get, go, year, bad, start


In [11]:
component_weights = reg.coef_
for i, weight in enumerate(component_weights[:10]):
    print(f"Component {i+1}: Weight = {weight:.3f}")


Component 1: Weight = 0.130
Component 2: Weight = -0.239
Component 3: Weight = -0.257
Component 4: Weight = 0.141
Component 5: Weight = -0.107
Component 6: Weight = -0.582
Component 7: Weight = -0.412
Component 8: Weight = -0.223
Component 9: Weight = -0.001
Component 10: Weight = 0.393


# Sentiment analysis 

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from textblob import TextBlob
from scipy.sparse import hstack
import numpy as np

# Step 1: Vectorize the cleaned review text
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['review_clean'])  # df['review_clean'] should be preprocessed text

# # Step 2: Compute sentiment polarity for each review
# sentiment_scores = df['review_clean'].apply(lambda x: TextBlob(x).sentiment.polarity)
# sentiment_scores = sentiment_scores.values.reshape(-1, 1)  # Convert to column vector

# Step 3: Combine BoW and sentiment features
X_combined = hstack([X_bow, sentiment_scores])  # hstack keeps it in sparse format for efficiency

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, df['rating'], test_size=0.2, random_state=42)

# Step 5: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 6: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))


MSE: 8.850253626799049


# Vader Sentiment 

In [24]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
def sentiment_scores_column(series):
    analyzer = SentimentIntensityAnalyzer()
    return series.apply(lambda text: analyzer.polarity_scores(text)['compound']).values

# Step 1: Vectorize the cleaned review text
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['review_clean']) 

# Step 2: Compute sentiment scores for each review
sentiment_scores = sentiment_scores_column(df['review_clean'])
sentiment_scores = sentiment_scores.reshape(-1, 1)  

# Step 3: Combine BoW and sentiment features
X_combined = hstack([X_bow, sentiment_scores])  # hstack keeps it in sparse format for efficiency

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, df['rating'], test_size=0.2, random_state=42)

# Step 5: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 6: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))


MSE: 8.383217030233588


In [23]:
from gensim.models import Word2Vec
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.sparse import hstack

# Step 1: Compute sentiment scores for each review
def sentiment_scores_column(series):
    analyzer = SentimentIntensityAnalyzer()
    return series.apply(lambda text: analyzer.polarity_scores(text)['compound']).values

# 1. Tokenize reviews
df['tokens'] = df['review_clean'].apply(lambda x: x.split())

# 2. Train Word2Vec model
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

# 3. Fit TF-IDF
tfidf = TfidfVectorizer()
tfidf.fit(df['review_clean'])
tfidf_weights = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

# 4. Function to get TF-IDF weighted average Word2Vec
def tfidf_weighted_w2v(tokens, model, tfidf_weights, vector_size):
    vec = np.zeros(vector_size)
    weight_sum = 0
    for word in tokens:
        if word in model.wv and word in tfidf_weights:
            weight = tfidf_weights[word]
            vec += model.wv[word] * weight
            weight_sum += weight
    return vec / weight_sum if weight_sum > 0 else vec

# 5. Create document vectors
vector_size = w2v_model.vector_size
X_tfidf_w2v = np.vstack(df['tokens'].apply(lambda tokens: tfidf_weighted_w2v(tokens, w2v_model, tfidf_weights, vector_size)))

# 6. Compute sentiment scores
sentiment_scores = sentiment_scores_column(df['review_clean'])
sentiment_scores = sentiment_scores.reshape(-1, 1)  # Ensure correct shape

# 7. Combine Word2Vec vectors with sentiment score
X_combined = np.hstack([X_tfidf_w2v, sentiment_scores])

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, df['rating'], test_size=0.2, random_state=42)

# Step 5: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 6: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

MSE: 8.661149595958532


In [33]:
from sklearn.decomposition import PCA

# Step 7: Reduce dimensionality with PCA
n_components = 10  # You can adjust this
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_combined)

# OPTIONAL: View explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

# Step 8: Train/test split on PCA-transformed data
X_train, X_test, y_train, y_test = train_test_split(X_pca, df['rating'], test_size=0.2, random_state=42)

# Step 9: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 10: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))


Explained variance ratio: [0.14156205 0.09104212 0.07788627 0.06764253 0.06138101 0.0526627
 0.03595457 0.03336711 0.02744497 0.02197323]
MSE: 8.504994452753403


In [34]:
# Create feature names
feature_names = [f'w2v_{i}' for i in range(vector_size)] + ['sentiment']

# Choose component index (e.g. first component)
component_idx = 0
component_weights = pca.components_[component_idx]

# Get top 10 negative and positive features
sorted_indices = np.argsort(component_weights)
most_negative = [(feature_names[i], component_weights[i]) for i in sorted_indices[:10]]
most_positive = [(feature_names[i], component_weights[i]) for i in sorted_indices[-10:][::-1]]

# Display
print(f"\n🔻 Most Negative Features in PCA Component {component_idx + 1}:")
for feat, val in most_negative:
    print(f"{feat:20s} {val:.4f}")

print(f"\n🔺 Most Positive Features in PCA Component {component_idx + 1}:")
for feat, val in most_positive:
    print(f"{feat:20s} {val:.4f}")



🔻 Most Negative Features in PCA Component 1:
w2v_26               -0.1975
w2v_17               -0.1939
w2v_8                -0.1901
w2v_51               -0.1897
w2v_36               -0.1819
w2v_15               -0.1776
w2v_54               -0.1677
w2v_11               -0.1598
w2v_53               -0.1574
w2v_58               -0.1489

🔺 Most Positive Features in PCA Component 1:
w2v_22               0.2816
w2v_94               0.1870
w2v_74               0.1848
w2v_45               0.1838
w2v_21               0.1808
w2v_72               0.1599
w2v_34               0.1507
w2v_9                0.1375
w2v_35               0.1321
w2v_31               0.1140


In [31]:
pip show gensim

Name: gensim
Version: 4.3.3
Summary: Python framework for fast Vector Space Modelling
Home-page: https://radimrehurek.com/gensim/
Author: Radim Rehurek
Author-email: me@radimrehurek.com
License: LGPL-2.1-only
Location: /Users/youssefbenmansour/Desktop/HCML-NLP-Project/venv/lib/python3.12/site-packages
Requires: numpy, scipy, smart-open
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [41]:
print(df['review_clean'][44])

used to use depo provera for years then went through  yrs no period it then came back with a vengence had abnormally high amount of bleeding for  days with such severe cramping that it would often result in a missed day or  of work each month for months on end before talking to my dr shortly thereafter i  eventually got an endometrial biopsy i was told the endometrial tissue was hormonally confused was then put on lo loestrin fe i noticed a huge amount of positive difference within first month  still  presently light bleeding  minimal cramping for  days its like night  day i would highly recommend


# Julia version

In [1]:
from feature_extraction import feature_extraction
import pandas as pd

# Load data
df = pd.read_csv("/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv")
features = feature_extraction(df)

/Users/youssefbenmansour/Desktop/HCML-NLP-Project/venv/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.0)
  from scipy.sparse import csr_matrix, issparse


True None None None


: 